# 2. Конформные интервалы

**Основная идея:** калибровка интервалов на отложенных данных с гарантией покрытия при перестановочности.

В этом ноутбуке:
- Split conformal prediction руками (~10 строк);
- калибровка отдельно по каждому горизонту;
- statsforecast ConformalIntervals;
- обёртка CatBoost через conformalforecast;
- адаптивные методы для дрейфа (ACI, AgACI, NexCP).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA
from statsforecast.utils import ConformalIntervals

from conformalforecast import SplitConformal, ACI, AgACI, NexCP

from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit

sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 4)

## 0. Данные

Используем те же данные AEP, что и в предыдущем ноутбуке.

In [ ]:
# Загрузка и агрегация до дня
df_hour = pd.read_csv('../data/AEP_hourly.csv', parse_dates=['Datetime'])
df_hour.columns = ['timestamp', 'target']

df_day = df_hour.resample('D', on='timestamp').sum().reset_index()
df_day = df_day.rename(columns={'timestamp': 'ds', 'target': 'y'})

# Разбиение: train 80%, остаток для калибровки и теста
split_point = int(len(df_day) * 0.8)
calib_size = 200

train = df_day.iloc[:split_point].copy()
calib = df_day.iloc[split_point:split_point + calib_size].copy()
test = df_day.iloc[split_point + calib_size:].copy()

print(f"Train: {len(train)}, Calibration: {len(calib)}, Test: {len(test)}")

## 1. Split Conformal руками

Алгоритм:
1. Обучаем модель на train.
2. Прогнозируем на calibration, считаем абсолютные остатки (скоры).
3. Берём (⌈(n+1)(1−α)⌉/n)-квантиль скоров — это поправка Q.
4. Интервал на тесте: [ŷ − Q, ŷ + Q].

**Гарантия:** при перестановочности (train ∪ calib ∪ test) покрытие ≥ 1−α.

In [ ]:
# Модель
train_sf = train.copy()
train_sf['unique_id'] = 'AEP'

sf = StatsForecast(models=[AutoETS(season_length=7)], freq='D', n_jobs=1)
sf.fit(train_sf[['unique_id', 'ds', 'y']])

In [ ]:
# Калибровочные скоры
horizon = 14
calib_forecasts = []
calib_actuals = []

for i in range(0, len(calib) - horizon, horizon):
    fc = sf.predict(h=horizon, level=None)
    actual = calib['y'].iloc[i:i+horizon].values
    calib_forecasts.append(fc['AutoETS'].values)
    calib_actuals.append(actual)

    # Обновляем модель (re-fit)
    new_train = pd.concat([train_sf, calib.iloc[:i+horizon].assign(unique_id='AEP')])
    sf.fit(new_train[['unique_id', 'ds', 'y']])

calib_forecasts = np.concatenate(calib_forecasts)
calib_actuals = np.concatenate(calib_actuals)

# Скоры
scores = np.abs(calib_actuals - calib_forecasts)

In [ ]:
# Квантиль с поправкой (n+1)
alpha = 0.05
n = len(scores)
q_level = np.ceil((n + 1) * (1 - alpha)) / n
Q = np.quantile(scores, q_level)

print(f"Калибровочных точек: {n}")
print(f"Квантиль уровня {q_level:.4f}: Q = {Q:.1f}")

In [ ]:
# Прогноз на тесте
test_forecast = sf.predict(h=len(test), level=None)
y_pred_test = test_forecast['AutoETS'].values
lower_cp = y_pred_test - Q
upper_cp = y_pred_test + Q

In [ ]:
# Визуализация первых 60 точек
plt.figure(figsize=(14, 4))
plt.plot(range(60), test['y'].values[:60], 'o-', label='Факт', markersize=3)
plt.plot(range(60), y_pred_test[:60], 's-', label='Прогноз', markersize=3)
plt.fill_between(range(60), lower_cp[:60], upper_cp[:60], alpha=0.3, label='95% Split Conformal')
plt.legend()
plt.title('Split Conformal Prediction руками')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Эмпирическое покрытие
coverage = ((test['y'].values >= lower_cp) & (test['y'].values <= upper_cp)).mean()
print(f"Покрытие на тесте: {coverage:.3f} (номинал 0.950)")

**Важно:** интервал одинаковой ширины для всех горизонтов — это упрощение. В реальности нужна калибровка отдельно по каждому h.

## 2. Калибровка по горизонтам

Скоры на калибровке группируем по горизонту h=1, h=2, ..., h=14.

In [ ]:
# Пересчитываем скоры с учётом горизонта
horizon_scores = {h: [] for h in range(1, horizon + 1)}

# Переделаем калибровку
sf.fit(train_sf[['unique_id', 'ds', 'y']])

for i in range(0, len(calib) - horizon, horizon):
    fc = sf.predict(h=horizon, level=None)
    actual = calib['y'].iloc[i:i+horizon].values

    for h_idx in range(min(horizon, len(actual))):
        score = np.abs(actual[h_idx] - fc['AutoETS'].iloc[h_idx])
        horizon_scores[h_idx + 1].append(score)

    # Обновляем модель
    new_train = pd.concat([train_sf, calib.iloc[:i+horizon].assign(unique_id='AEP')])
    sf.fit(new_train[['unique_id', 'ds', 'y']])

In [ ]:
# Квантили по каждому горизонту
Q_by_h = {}
for h in range(1, horizon + 1):
    if len(horizon_scores[h]) > 0:
        n_h = len(horizon_scores[h])
        q_level_h = np.ceil((n_h + 1) * (1 - alpha)) / n_h
        Q_by_h[h] = np.quantile(horizon_scores[h], q_level_h)
    else:
        Q_by_h[h] = Q  # fallback

In [ ]:
# Визуализация роста Q с горизонтом
plt.figure(figsize=(10, 4))
plt.plot(list(Q_by_h.keys()), list(Q_by_h.values()), 'o-')
plt.axhline(Q, color='red', linestyle='--', label='Общий Q (без учёта h)')
plt.xlabel('Горизонт')
plt.ylabel('Конформная поправка Q')
plt.title('Рост поправки с горизонтом')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Q(h=1) = {Q_by_h[1]:.1f}, Q(h=14) = {Q_by_h[14]:.1f}")

## 3. statsforecast ConformalIntervals

Библиотека реализует калибровку по горизонтам и скользящим окнам.

In [ ]:
# Подготовка данных в long-формате
df_long = df_day.copy()
df_long['unique_id'] = 'AEP'

# Обучение модели + конформные интервалы
sf_conf = StatsForecast(
    models=[AutoETS(season_length=7)],
    freq='D',
    n_jobs=1
)

# Разбиение для statsforecast: весь ряд кроме последних test
train_conf = df_long.iloc[:split_point + calib_size]

sf_conf.fit(train_conf[['unique_id', 'ds', 'y']])

# Прогноз с конформными интервалами
# ConformalIntervals требует n_windows калибровочных окон
forecast_conf = sf_conf.predict(
    h=len(test),
    level=[95],
    prediction_intervals=ConformalIntervals(h=horizon, n_windows=10)
)

In [ ]:
y_pred_conf = forecast_conf['AutoETS'].values
lower_conf = forecast_conf['AutoETS-lo-95'].values
upper_conf = forecast_conf['AutoETS-hi-95'].values

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(range(60), test['y'].values[:60], 'o-', label='Факт', markersize=3)
plt.fill_between(range(60), lower_conf[:60], upper_conf[:60], alpha=0.3, label='95% ConformalIntervals')
plt.legend()
plt.title('statsforecast ConformalIntervals')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

coverage_conf = ((test['y'].values >= lower_conf) & (test['y'].values <= upper_conf)).mean()
print(f"Покрытие: {coverage_conf:.3f}")

## 4. Обёртка CatBoost через conformalforecast

Конформные интервалы модельно-независимы: можем обернуть любую модель, включая бустинг.

In [ ]:
# Признаки: лаги и календарь
def make_features(df, lags=list(range(14, 28))):
    df = df.copy()
    df['dow'] = df['ds'].dt.dayofweek
    df['month'] = df['ds'].dt.month

    for lag in lags:
        df[f'lag_{lag}'] = df['y'].shift(lag)

    df = df.dropna()
    return df

In [ ]:
df_feats = make_features(df_day)

# Разбиение
X_train = df_feats.iloc[:split_point][['dow', 'month'] + [c for c in df_feats.columns if c.startswith('lag_')]]
y_train = df_feats.iloc[:split_point]['y']

X_calib = df_feats.iloc[split_point:split_point + calib_size][X_train.columns]
y_calib = df_feats.iloc[split_point:split_point + calib_size]['y']

X_test = df_feats.iloc[split_point + calib_size:][X_train.columns]
y_test = df_feats.iloc[split_point + calib_size:]['y']

In [ ]:
# Модель
cb = CatBoostRegressor(iterations=100, depth=4, verbose=False, random_state=42)
cb.fit(X_train, y_train)

# Прогнозы
y_pred_cb_calib = cb.predict(X_calib)
y_pred_cb_test = cb.predict(X_test)

In [ ]:
# Split Conformal через conformalforecast
cp = SplitConformal()
cp.fit(y_calib.values, y_pred_cb_calib)

lower_cb, upper_cb = cp.predict(y_pred_cb_test, alpha=0.05)

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(range(60), y_test.values[:60], 'o-', label='Факт', markersize=3)
plt.plot(range(60), y_pred_cb_test[:60], 's-', label='CatBoost', markersize=3)
plt.fill_between(range(60), lower_cb[:60], upper_cb[:60], alpha=0.3, label='95% Split Conformal')
plt.legend()
plt.title('CatBoost + Split Conformal')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

coverage_cb = ((y_test.values >= lower_cb) & (y_test.values <= upper_cb)).mean()
print(f"Покрытие: {coverage_cb:.3f}")

## 5. Адаптивные методы для дрейфа

Если данные нестационарны (дрейф среднего, гетероскедастичность), split conformal теряет покрытие.
Адаптивные методы онлайн-обновляют поправку Q.

### ACI (Adaptive Conformal Inference)

Обновляет α в зависимости от ошибок покрытия: если факт выходит за интервал, уменьшаем α (расширяем интервал).

In [ ]:
# Генерация синтетики с дрейфом для демонстрации
np.random.seed(42)
n_demo = 500
t_demo = np.arange(n_demo)
y_demo = 100 + 0.1 * t_demo + 10 * np.sin(2 * np.pi * t_demo / 50) + np.random.normal(0, 5, n_demo)

# Простая модель: скользящее среднее
window = 20
y_pred_demo = pd.Series(y_demo).rolling(window, min_periods=1).mean().shift(1).fillna(y_demo[0])

In [ ]:
# Split conformal (статический)
n_calib_demo = 100
scores_demo = np.abs(y_demo[:n_calib_demo] - y_pred_demo[:n_calib_demo])
Q_demo = np.quantile(scores_demo, 0.95)

lower_split_demo = y_pred_demo[n_calib_demo:] - Q_demo
upper_split_demo = y_pred_demo[n_calib_demo:] + Q_demo

In [ ]:
# ACI
aci = ACI(gamma=0.01)  # скорость адаптации
aci.fit(y_demo[:n_calib_demo], y_pred_demo[:n_calib_demo])

lower_aci, upper_aci = [], []
for i in range(n_calib_demo, n_demo):
    lo, up = aci.predict(y_pred_demo[i:i+1], alpha=0.05)
    lower_aci.append(lo[0])
    upper_aci.append(up[0])

    # Обновляем ACI с фактом
    aci.update(y_demo[i:i+1], y_pred_demo[i:i+1])

lower_aci = np.array(lower_aci)
upper_aci = np.array(upper_aci)

In [ ]:
# Визуализация
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Split
axes[0].plot(t_demo[n_calib_demo:], y_demo[n_calib_demo:], 'o-', label='Факт', markersize=2)
axes[0].fill_between(t_demo[n_calib_demo:], lower_split_demo, upper_split_demo, alpha=0.3, label='Split Conformal')
axes[0].set_title('Split Conformal (статический)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ACI
axes[1].plot(t_demo[n_calib_demo:], y_demo[n_calib_demo:], 'o-', label='Факт', markersize=2)
axes[1].fill_between(t_demo[n_calib_demo:], lower_aci, upper_aci, alpha=0.3, label='ACI', color='orange')
axes[1].set_title('ACI (адаптивный)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
coverage_split_demo = ((y_demo[n_calib_demo:] >= lower_split_demo) & (y_demo[n_calib_demo:] <= upper_split_demo)).mean()
coverage_aci_demo = ((y_demo[n_calib_demo:] >= lower_aci) & (y_demo[n_calib_demo:] <= upper_aci)).mean()

print(f"Покрытие Split: {coverage_split_demo:.3f}")
print(f"Покрытие ACI: {coverage_aci_demo:.3f}")
print("ACI удерживает покрытие ближе к номиналу при дрейфе.")

## Итог

- **Split Conformal**: калибровка на отложенных данных, гарантия покрытия при перестановочности.
- **Калибровка по горизонтам**: интервал расширяется с горизонтом.
- **Модельно-независимость**: оборачиваем любую модель (ETS, CatBoost, ...).
- **Адаптивные методы (ACI, AgACI, NexCP)**: онлайн-обновление для нестационарных рядов.
- **Маргинальное покрытие**: гарантия на среднее по тесту, а не условное (для каждой точки отдельно).

**Следующий ноутбук:** квантильная регрессия (QAR, CatBoost Quantile, CQR).